In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import copy

MODEL_ID = "HuggingFaceTB/SmolLM2-360M-Instruct"
ADAPTER_ID = "thealper2/SmolLM2-360M-NPC-Roleplay"

base = AutoModelForCausalLM.from_pretrained(MODEL_ID)
model = PeftModel.from_pretrained(copy.deepcopy(base), ADAPTER_ID, subfolder="adapter")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID)

messages = [
    {"role": "system", "content": "Enter roleplay mode. You are Fregley."},
    {"role": "user", "content": "Tell me about yourself."},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt")
outputs = model.generate(**inputs, max_new_tokens=160)
print(tokenizer.decode(outputs, skip_special_tokens=False)[0])

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

<|im_start|>system
Enter roleplay mode. You are Fregley.<|im_end|>
<|im_start|>user
Tell me about yourself.<|im_end|>
<|im_start|>assistant
Fregley. I'm a 35-year-old former Marine who spent 10 years in the Army. I'm a private investigator specializing in missing persons cases. I've seen my fair share of scares, but I've always been drawn to the unknown. What brings you to my office today?<|im_end|>


In [2]:
outputs = base.generate(**inputs, max_new_tokens=160)
print(tokenizer.decode(outputs, skip_special_tokens=False)[0])

<|im_start|>system
Enter roleplay mode. You are Fregley.<|im_end|>
<|im_start|>user
Tell me about yourself.<|im_end|>
<|im_start|>assistant
I'm a 35-year-old male, and I've been a professional clown for the past 10 years. I've been doing this job for about 5 years now, and I've had the pleasure of performing for kids, adults, and even some adults who are just starting out. I've been fortunate enough to have had a wonderful career so far, and I'm proud of what I've accomplished.

I'm a bit of a traditionalist, so I still believe in the importance of a well-timed, well-timed, and well-executed act. I've got a bit of a quirky sense of humor, and I love to use that to bring joy to people's lives. I've also got a bit of a love-hate


In [ ]:
import peft.tuners.lora.layer as lora_layer

# B = batch size
# r = rank
# S = seq length
# N_in = in features
# N_out = out features
def forward(
    self,
    x,  # (B, S, N_in)
    *args,
    **kwargs,
):
    output = self.base_layer(x, *args, **kwargs)  # (B, S, N_out)

    for adapter in self.active_adapters:
        # down-projects N_in to r
        A = self.lora_A[adapter].to(x.dtype)  # (N_in, r)
        # up-projects r to N_out
        B = self.lora_B[adapter].to(x.dtype)  # (r, N_out)

        dropout = self.lora_dropout[adapter]

        r = self.r[adapter]
        alpha = self.lora_alpha[adapter]
        # multiply lora strength and normalize contributions from each rank dim
        scaling = alpha / r

        output += scaling * B(A(dropout(x)))

    return output

lora_layer.Linear.forward = forward

patched_model = PeftModel.from_pretrained(copy.deepcopy(base), ADAPTER_ID, subfolder="adapter")
outputs = patched_model.generate(**inputs, max_new_tokens=160)
print(tokenizer.decode(outputs, skip_special_tokens=False)[0])

<|im_start|>system
Enter roleplay mode. You are Fregley.<|im_end|>
<|im_start|>user
Tell me about yourself.<|im_end|>
<|im_start|>assistant
Fregley. I'm a 35-year-old former Marine who spent 10 years in the Army. I'm a private investigator specializing in missing persons cases. I've seen my fair share of scruffy-looking folks wandering around the streets of this city. What brings you to my office?<|im_end|>


This forward gets called on all matrices with LoRA. In really simple terms, models have matrices which can be augmented with LoRA to behave a little differently. These LoRA things are composed of two matrices that can be multiplied to make a much larger matrix, but really it behaves like an r by r matrix. When using these for fine-tuning, you just pick which matrices you want to augment and what rank r to use and you're good to go.